# 02 Cleaning
`src/clean.py` is independent of the SQL staging layer; this notebook re-runs it and compares the two.

In [1]:
import json, pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
facts = json.load(open("reports/facts.json", encoding="utf-8"))
from src import clean
df, log = clean.build()
log

clean: 7,043 customers, churn rate 0.2654; 5 logged decisions


,step,action,rows_before,rows_after,rows_affected,decision
0,1,Drop duplicate customerID,7043,7043,0,One row per customer is required for churn rates.
1,2,Set blank TotalCharges to 0,7043,7043,11,All 11 blanks have tenure 0: new customers not...
2,3,Encode Churn Yes/No as 1/0,7043,7043,1869,Target for rates and models.
3,4,Add tenure and monthly-charge bands,7043,7043,0,Readable segments for reporting; models use th...
4,5,Keep TotalCharges that differ from tenure x Mo...,7043,7043,59,Plan changes over a customer's life explain th...


In [2]:
from src.warehouse import connect
with connect(read_only=True) as con:
    sql = con.execute('SELECT COUNT(*) n, AVG(churned) churn_rate, SUM(monthly_charges*churned) mrr_lost, SUM(total_charges) total FROM stg_customers').df()
py = pd.DataFrame({'n': [len(df)], 'churn_rate': [df.churned.mean()], 'mrr_lost': [(df.monthly_charges*df.churned).sum()], 'total': [df.total_charges.sum()]})
pd.concat({'SQL stg_customers': sql, 'Python customers_clean': py})

,,n,churn_rate,mrr_lost,total
SQL stg_customers,0,7043,0.26537,139130.85,16056168.7
Python customers_clean,0,7043,0.26537,139130.85,16056168.7
